# Financial Inclusion and Digital Financial Behaviour
## 1. Data Loading and Preparation

### 1.1 Load and Inspect the Dataset

In [2]:
import pandas as pd
import numpy as np

file_path = "data/GlobalFindexDatabase2025.xlsx"

df = pd.read_excel(file_path, sheet_name="Data")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (8578, 466)


,countrynewwb,codewb,year,pop_adult,regionwb24_hi,incomegroupwb24,group,group2,account.t.d,fiaccount.t.d,...,con32d,con32d.s,con32e,con32e.s,con32f,con32f.s,con32g,con32g.s,con32h,con32h.s
0,Economy,Economy code,Year,Adult population,Region,Income group,Demographic group,Demographic sub-group,"Account (%, age 15+)",Bank or similar financial institution account ...,...,Main reason for not owning a smartphone: diffi...,Main reason for not owning a smartphone: diffi...,Main reason for not owning a smartphone: famil...,Main reason for not owning a smartphone: famil...,Main reason for not owning a smartphone: safet...,Main reason for not owning a smartphone: safet...,Main reason for not owning a smartphone: uses ...,Main reason for not owning a smartphone: uses ...,Main reason for not owning a smartphone: no ne...,Main reason for not owning a smartphone: no ne...
1,Afghanistan,AFG,2011,14575546,South Asia (excluding high income),Low income,all,all,0.09005,0.09005,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Albania,ALB,2011,2281010,Europe & Central Asia (excluding high income),Upper middle income,all,all,0.282681,0.282681,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Algeria,DZA,2011,26251587,Middle East & North Africa (excluding high inc...,Lower middle income,all,all,0.332861,0.332861,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Angola,AGO,2011,12779501,Sub-Saharan Africa (excluding high income),Lower middle income,all,all,0.392035,0.392035,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
#Structure of data before cleaning

df[["countrynewwb", "year", "group", "group2", "incomegroupwb24"]].head(10)

,countrynewwb,year,group,group2,incomegroupwb24
0,Economy,Year,Demographic group,Demographic sub-group,Income group
1,Afghanistan,2011,all,all,Low income
2,Albania,2011,all,all,Upper middle income
3,Algeria,2011,all,all,Lower middle income
4,Angola,2011,all,all,Lower middle income
5,Argentina,2011,all,all,Upper middle income
6,Armenia,2011,all,all,Upper middle income
7,Australia,2011,all,all,High income
8,Austria,2011,all,all,High income
9,Azerbaijan,2011,all,all,Upper middle income


### 1.2 Filter the dataset by Year,Population group and income classification

In [4]:
# Data filtering: select 2024 country-level records

data_2024 = df[
    (df["year"].astype(str) == "2024") &
    (df["group"] == "all") &
    (df["group2"] == "all")
].copy()

print("2024 country-level records:", data_2024.shape[0])

2024 country-level records: 153


In [5]:
# select developing economies

income_groups = [
    "Low income",
    "Lower middle income",
    "Upper middle income"
]

developing_data = data_2024[data_2024["incomegroupwb24"].isin(income_groups)].copy()

print("Developing economies:", developing_data.shape[0])

Developing economies: 93


###  1.3  Select Financial Indicators

In [6]:
#Select the 11 financial indicators

financial_indicators = [
    "fiaccount.t.d",  #Fianancial institution account
    "fin2.t.d",       #Debit card ownership
    "fin10",          #Credit card ownership
    "merchant.pay",   #Digital merchant payments
    "fin27a",         #online purchase payments
    "save.any.t.d",   #saved any money
    "fin17a",         #saved at financial instituion
    "fin22a",         #borrowed from financial institution
    "fin22b",         #borrowed from family or friends
    "internet",       #internet usage
]

financial_data = developing_data[["countrynewwb", "account.t.d"] + financial_indicators].copy()

financial_data["mobile_money_only"] = (financial_data["account.t.d"] - financial_data["fiaccount.t.d"])

print("Dataset shape:", financial_data.shape)
financial_data.head()

Dataset shape: (93, 13)


,countrynewwb,account.t.d,fiaccount.t.d,fin2.t.d,fin10,merchant.pay,fin27a,save.any.t.d,fin17a,fin22a,fin22b,internet,mobile_money_only
572,Albania,0.460693,0.460693,0.327838,0.073574,0.110836,0.032812,0.285807,0.147581,0.075687,0.249029,0.828402,0.0
573,Algeria,0.352901,0.352901,0.221997,0.052144,0.070034,0.033282,0.380609,0.16552,0.036918,0.269959,0.829104,0.0
574,Argentina,0.817442,0.72899,0.640681,0.299115,0.601935,0.319846,0.480498,0.231229,0.091076,0.245174,0.881293,0.088453
575,Armenia,0.713735,0.705333,0.437131,0.07688,0.338039,0.197056,0.419728,0.091174,0.203019,0.263069,0.875233,0.008402
578,Azerbaijan,0.563047,0.563047,0.492829,0.13382,0.342396,0.111681,0.26657,0.169616,0.191888,0.357088,0.863524,0.0



### 1.4 Identify Variable Types and Measurement Scales

Before analysing our data, we need to understand the type of variables we have and how they are measured.

Our dataset contains country names and financial indicators. The country names are nominal variables because they identify countries without any particular order.

The financial indicators are quantitative variables measured on a ratio scale. They show the proportion of adults who own or use different financial services.

| Variable | Measurement Scale |
|---|---|
| Country | Nominal |
| Financial institution account ownership | Ratio |
| Mobile-money-only account ownership | Ratio |
| Debit card ownership | Ratio |
| Credit card ownership | Ratio |
| Digital merchant payments | Ratio |
| Online purchase payments | Ratio |
| Saved any money | Ratio |
| Savings at a financial institution | Ratio |
| Borrowing from a financial institution | Ratio |
| Borrowing from family or friends | Ratio |
| Internet usage | Ratio |

The financial indicators have values between 0 and 1. For example, a value of 0.60 for debit card ownership means that 60% of adults own a debit card.

We also checked how Python stored these variables. The financial indicators were initially stored as `object`, so we converted them to `float64` to make sure Python recognises them as numerical values.




In [12]:
financial_data.dtypes

countrynewwb            str
account.t.d          object
fiaccount.t.d        object
fin2.t.d             object
fin10                object
merchant.pay         object
fin27a               object
save.any.t.d         object
fin17a               object
fin22a               object
fin22b               object
internet             object
mobile_money_only    object
dtype: object

In [14]:
# Convert financial indicators to numerical data types
numeric_columns = financial_data.columns.drop("countrynewwb")

financial_data[numeric_columns] = financial_data[numeric_columns].apply(
    pd.to_numeric, errors="coerce"
)

# Verify the data types after conversion
financial_data.dtypes

countrynewwb             str
account.t.d          float64
fiaccount.t.d        float64
fin2.t.d             float64
fin10                float64
merchant.pay         float64
fin27a               float64
save.any.t.d         float64
fin17a               float64
fin22a               float64
fin22b               float64
internet             float64
mobile_money_only    float64
dtype: object

### 1.5   Check and Handle Missing Values

In [15]:
financial_data.isna().sum()

countrynewwb         0
account.t.d          0
fiaccount.t.d        0
fin2.t.d             0
fin10                0
merchant.pay         1
fin27a               1
save.any.t.d         1
fin17a               1
fin22a               1
fin22b               1
internet             0
mobile_money_only    0
dtype: int64

In [16]:
#identify economies with missing values
missing_rows = financial_data[financial_data.isna().any(axis=1)]

print(missing_rows[["countrynewwb"]].to_string(index=False))
print("Economies affected:", len(missing_rows))

      countrynewwb
Russian Federation
Economies affected: 1


In [17]:
financial_data_clean = financial_data.dropna().copy()

print("Before cleaning:", financial_data.shape[0])
print("After cleaning:", financial_data_clean.shape[0])

Before cleaning: 93
After cleaning: 92


### 1.6  Rename Variables for Readability

In [18]:
financial_data_clean = financial_data_clean.rename(columns={
    "countrynewwb" : "country",
    "account.t.d": "account_ownership",
    "fiaccount.t.d": "bank_account",
    "fin2.t.d": "debit_card",
    "fin10": "credit_card",
    "merchant.pay": "digital_merchant_payment",
    "fin27a": "online_purchase_payment",
    "save.any.t.d": "saved_any_money",
    "fin17a": "formal_savings",
    "fin22a": "formal_borrowing",
    "fin22b": "family_friends_borrowing",
    "internet": "internet_usage"
})

financial_data_clean.head()

,country,account_ownership,bank_account,debit_card,credit_card,digital_merchant_payment,online_purchase_payment,saved_any_money,formal_savings,formal_borrowing,family_friends_borrowing,internet_usage,mobile_money_only
572,Albania,0.460693,0.460693,0.327838,0.073574,0.110836,0.032812,0.285807,0.147581,0.075687,0.249029,0.828402,0.000000
573,Algeria,0.352901,0.352901,0.221997,0.052144,0.070034,0.033282,0.380609,0.165520,0.036918,0.269959,0.829104,0.000000
574,Argentina,0.817442,0.728990,0.640681,0.299115,0.601935,0.319846,0.480498,0.231229,0.091076,0.245174,0.881293,0.088453
575,Armenia,0.713735,0.705333,0.437131,0.076880,0.338039,0.197056,0.419728,0.091174,0.203019,0.263069,0.875233,0.008402
578,Azerbaijan,0.563047,0.563047,0.492829,0.133820,0.342396,0.111681,0.266570,0.169616,0.191888,0.357088,0.863524,0.000000


### 1.7  Prepare the Final Dataset

In [19]:
# the 11 indicators for multivariate analysis
indicator_columns = [
    "bank_account",
    "mobile_money_only",
    "debit_card",
    "credit_card",
    "digital_merchant_payment",
    "online_purchase_payment",
    "saved_any_money",
    "formal_savings",
    "formal_borrowing",
    "family_friends_borrowing",
    "internet_usage"
]

# Keep country names alongside the financial indicators
analysis_data = financial_data_clean[
    ["country"] + indicator_columns
].copy()

print("Final dataset shape:", analysis_data.shape)
print("Missing values:", analysis_data.isna().sum().sum())

analysis_data.head()

Final dataset shape: (92, 12)
Missing values: 0


,country,bank_account,mobile_money_only,debit_card,credit_card,digital_merchant_payment,online_purchase_payment,saved_any_money,formal_savings,formal_borrowing,family_friends_borrowing,internet_usage
572,Albania,0.460693,0.000000,0.327838,0.073574,0.110836,0.032812,0.285807,0.147581,0.075687,0.249029,0.828402
573,Algeria,0.352901,0.000000,0.221997,0.052144,0.070034,0.033282,0.380609,0.165520,0.036918,0.269959,0.829104
574,Argentina,0.728990,0.088453,0.640681,0.299115,0.601935,0.319846,0.480498,0.231229,0.091076,0.245174,0.881293
575,Armenia,0.705333,0.008402,0.437131,0.076880,0.338039,0.197056,0.419728,0.091174,0.203019,0.263069,0.875233
578,Azerbaijan,0.563047,0.000000,0.492829,0.133820,0.342396,0.111681,0.266570,0.169616,0.191888,0.357088,0.863524
